In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Data Loading and Preprocessing
Read the dataset, center the hand coordinates to the wrist, then normalize them using the distance to the midlle finger.
Encode string labels to integers as neural networks require numerical inputs.
Normalization ensures the model is scale and translation invariant, preventing errors from different hand sizes or positions.
Finally do the dataSplitting, it is required to evaluate generalization on unseen data.

In [2]:
# 1. Load the dataset
df = pd.read_csv('hand_landmarks_data.csv')

# 2. Recenter all points to the wrist (x1, y1)
for i in range(2, 22):
    df[f'x{i}'] = df[f'x{i}'] - df['x1']
    df[f'y{i}'] = df[f'y{i}'] - df['y1']

# 3. Calculate distance to the mid-finger tip for scaling
dist = np.sqrt(df['x13']**2 + df['y13']**2)
dist = np.where(dist == 0, 1e-5, dist) # Prevent division by zero

# 4. Scale all coordinates using the calculated distance
for i in range(2, 22):
    df[f'x{i}'] = df[f'x{i}'] / dist
    df[f'y{i}'] = df[f'y{i}'] / dist

# Set wrist coordinates to exactly 0
df['x1'] = 0.0
df['y1'] = 0.0

# 5. Separate features (X) and labels (y)
X = df.drop('label', axis=1).values
y = df['label'].values

# 6. Convert string labels into numbers
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
num_classes = len(encoder.classes_)

# 7. Split data into Train (70%), Validation (15%), and Test (15%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y_encoded, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

# PyTorch Datasets & DataLoaders
Convert NumPy arrays into PyTorch tensors and map them to DataLoaders.
Because PyTorch models process tensors, not arrays.
DataLoaders are required to feed data into the model in manageable batches and shuffle training data to prevent the model from memorizing the order.

In [3]:
# Define a custom Dataset class for PyTorch
class GestureDataset(Dataset):
    def __init__(self, features, labels):
        # Convert arrays to PyTorch tensors
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Create dataset objects
train_dataset = GestureDataset(X_train, y_train)
val_dataset = GestureDataset(X_val, y_val)
test_dataset = GestureDataset(X_test, y_test)

# Create DataLoaders to handle batching (32 samples per batch)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Model Architecture
Define a Feedforward Neural Network with 3 linear layers, ReLU activation, and Dropout.
Linear layers process the 1D structured coordinate vectors.
ReLU handles non-linear relationships.
Dropout prevents overfitting by randomly disabling neurons during training.

In [4]:
# Define the Neural Network architecture
class GestureNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(GestureNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128) # First hidden layer
        self.relu = nn.ReLU()                 # Activation function
        self.dropout = nn.Dropout(0.2)        # Regularization to prevent overfitting
        self.fc2 = nn.Linear(128, 64)         # Second hidden layer
        self.fc3 = nn.Linear(64, num_classes) # Output layer

    def forward(self, x):
        # Define how data passes through the network
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x) 
        return x

# Initialize the model (63 input features = 21 landmarks * 3 coordinates)
model = GestureNN(input_size=63, num_classes=num_classes)

# Training Loop
Execute the forward pass, calculate loss using CrossEntropy and update weights via backpropagation using the Adam optimizer.
Monitor validation loss simultaneously.
This is the core mechanism that forces the model to learn the mapping between features and labels iteratively over 30 epochs.

In [5]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30
for epoch in range(epochs):
    # Set model to training mode
    model.train()
    train_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()            # Clear old gradients
        outputs = model(inputs)          # Forward pass: get predictions
        loss = criterion(outputs, labels) # Calculate error
        loss.backward()                  # Backward pass: calculate gradients
        optimizer.step()                 # Update model weights
        train_loss += loss.item()
    
    # Validation phase
    model.eval() # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad(): # Disable gradient tracking for speed and memory
        for inputs, labels in val_loader:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Get the predicted class
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    # Print progress every 5 epochs
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss/len(train_loader):.4f} - Val Loss: {val_loss/len(val_loader):.4f} - Val Acc: {100 * correct / total:.2f}%")

Epoch 1/30 - Train Loss: 1.3316 - Val Loss: 0.6926 - Val Acc: 79.78%
Epoch 5/30 - Train Loss: 0.1710 - Val Loss: 0.1409 - Val Acc: 96.63%
Epoch 10/30 - Train Loss: 0.0951 - Val Loss: 0.1000 - Val Acc: 97.66%
Epoch 15/30 - Train Loss: 0.0732 - Val Loss: 0.0857 - Val Acc: 97.98%
Epoch 20/30 - Train Loss: 0.0627 - Val Loss: 0.0808 - Val Acc: 98.39%
Epoch 25/30 - Train Loss: 0.0567 - Val Loss: 0.0774 - Val Acc: 98.47%
Epoch 30/30 - Train Loss: 0.0500 - Val Loss: 0.0782 - Val Acc: 98.47%


# Evaluation
Run the trained model on the unseen test set, calculate the overall accuracy score, and generate a classification report for per-class metrics.
Confirm that the model generalizes to new data and identifies if it struggles with specific gesture classes.

In [6]:
# Set model to evaluation mode
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        # Store predictions and actual labels
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

# Calculate and print accuracy
test_acc = accuracy_score(all_labels, all_preds)
print(f"\nOverall Test Accuracy: {test_acc * 100:.2f}%\n")

# Print detailed classification report
print("Classification Report on Test Data:")
print(classification_report(all_labels, all_preds, target_names=encoder.classes_))


Overall Test Accuracy: 98.49%

Classification Report on Test Data:
                 precision    recall  f1-score   support

           call       0.99      0.99      0.99       217
        dislike       1.00      1.00      1.00       202
           fist       0.98      0.99      0.99       152
           four       0.96      0.99      0.98       240
           like       1.00      0.99      1.00       233
           mute       0.98      0.96      0.97       152
             ok       0.99      0.99      0.99       258
            one       0.94      0.97      0.96       171
           palm       0.98      0.98      0.98       250
          peace       0.99      0.99      0.99       204
 peace_inverted       0.99      1.00      0.99       224
           rock       1.00      0.99      0.99       210
           stop       0.94      0.99      0.96       223
  stop_inverted       0.99      0.97      0.98       234
          three       1.00      0.96      0.98       227
         three2    